In [ ]:
!kaggle competitions download -c cafa-6-protein-function-prediction
!unzip /content/cafa-6-protein-function-prediction.zip

cafa-6-protein-function-prediction.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  /content/cafa-6-protein-function-prediction.zip
replace IA.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
# list all in tree
!apt install tree
!tree -L 2

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 1s (43.6 kB/s)
Selecting previously unselected package tree.
(Reading database ... 121713 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...
.
├── cafa-6-protein-function-prediction.zip
├── IA.tsv
├── models_ckpt
│   ├── model_fold_0.pt
│   ├── model_fold_1.pt
│   ├── model_fold_2.pt
│   ├── model_fold_3.pt
│   └── model_fold_4.pt
├── sample_data
│   ├── anscombe.json
│   ├── california_housing_test.csv
│   ├── california_ho

In [ ]:
!cat submission_example.csv

In [ ]:
!pip install obonet

In [ ]:
# Install dependencies for Graph and Bio-processing
!pip install -q torch-geometric obonet biopython peft

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, DenseGCNConv
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import obonet
import networkx as nx
from Bio import SeqIO
import math


In [ ]:
import torch
from transformers import EsmModel, AutoTokenizer
from Bio import SeqIO
from tqdm.auto import tqdm
import os
import numpy as np
import pandas as pd

import torch._dynamo
torch._dynamo.config.allow_unspec_int_on_nn_module = True
torch._dynamo.config.suppress_errors = True

MODEL_NAME = 'facebook/esm2_t33_650M_UR50D'
SEQ_PATH = 'Train/train_sequences.fasta'
TEST_PATH = 'Test/testsuperset.fasta'
SAVE_DIR = 'embeddings_650M'
BATCH_SIZE = 128

os.makedirs(SAVE_DIR, exist_ok=True)
device = 'cuda'

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = EsmModel.from_pretrained(MODEL_NAME).to(device)
model = torch.compile(model)
model.eval()

def extract_fast_sorted(fasta_path, prefix, limit=None):
    print(f"Parsing {prefix}...")
    data = []
    for i, record in enumerate(SeqIO.parse(fasta_path, "fasta")):
        if limit and i >= limit: break
        data.append({
            'id': record.id.split("|")[1] if "|" in record.id else record.id,
            'seq': str(record.seq),
            'len': len(record.seq)
        })

    data.sort(key=lambda x: x['len'], reverse=True)

    print(f"Extracting {len(data)} sequences (Sorted Batching)...")

    for i in tqdm(range(0, len(data), BATCH_SIZE)):
        batch = data[i : i+BATCH_SIZE]
        batch_seqs = [x['seq'] for x in batch]
        batch_ids = [x['id'] for x in batch]

        inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)

        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1).cpu()

        for b in range(len(batch_ids)):
            torch.save(embeddings[b].clone(), os.path.join(SAVE_DIR, f"{prefix}_{batch_ids[b]}.pt"))

extract_fast_sorted(SEQ_PATH, "train", limit=None)
extract_fast_sorted(TEST_PATH, "test", limit=None)
print("Extraction Complete!")

Loading facebook/esm2_t33_650M_UR50D...


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parsing train...
Extracting 82404 sequences (Sorted Batching)...


  0%|          | 0/644 [00:00<?, ?it/s]

W1201 18:21:57.101000 1900 torch/fx/experimental/symbolic_shapes.py:6833] [0/2] _maybe_guard_rel() was called on non-relation expression Eq(s52, s92) | Eq(s92, 1)
W1201 18:23:23.870000 1900 torch/fx/experimental/symbolic_shapes.py:6833] [0/3] _maybe_guard_rel() was called on non-relation expression Eq(s52, s92) | Eq(s92, 1)
W1201 18:26:32.831000 1900 torch/fx/experimental/symbolic_shapes.py:6833] [0/4] _maybe_guard_rel() was called on non-relation expression Eq(s52, s92) | Eq(s92, 1)
W1201 18:28:13.366000 1900 torch/fx/experimental/symbolic_shapes.py:6833] [0/5] _maybe_guard_rel() was called on non-relation expression Eq(s52, s92) | Eq(s92, 1)
W1201 18:28:30.731000 1900 torch/fx/experimental/symbolic_shapes.py:6833] [0/6] _maybe_guard_rel() was called on non-relation expression Eq(s52, s92) | Eq(s92, 1)
W1201 18:28:48.020000 1900 torch/fx/experimental/symbolic_shapes.py:6833] [0/7] _maybe_guard_rel() was called on non-relation expression Eq(s52, s92) | Eq(s92, 1)
W1201 18:29:06.419000 

Parsing test...
Extracting 224309 sequences (Sorted Batching)...


  0%|          | 0/1753 [00:00<?, ?it/s]

Extraction Complete!


In [ ]:
# zip /content/embeddings_650M
!zip -r embeddings_650M.zip /content/embeddings_650M

Streaming output truncated to the last 5000 lines.
  adding: content/embeddings_650M/train_Q9JLB5.pt (deflated 19%)
  adding: content/embeddings_650M/test_Q7Z5L4.pt (deflated 19%)
  adding: content/embeddings_650M/test_Q58CR3.pt (deflated 19%)
  adding: content/embeddings_650M/train_Q60775.pt (deflated 19%)
  adding: content/embeddings_650M/train_Q9XWL9.pt (deflated 19%)
  adding: content/embeddings_650M/test_P14083.pt (deflated 19%)
  adding: content/embeddings_650M/test_O70183.pt (deflated 19%)
  adding: content/embeddings_650M/test_Q8BMC4.pt (deflated 19%)
  adding: content/embeddings_650M/test_P17481.pt (deflated 19%)
  adding: content/embeddings_650M/test_B3LNJ6.pt (deflated 19%)
  adding: content/embeddings_650M/test_Q1AMT3.pt (deflated 19%)
  adding: content/embeddings_650M/train_O95571.pt (deflated 19%)
  adding: content/embeddings_650M/test_Q9FYG4.pt (deflated 19%)
  adding: content/embeddings_650M/train_Q9W0P8.pt (deflated 19%)
  adding: content/embeddings_650M/train_Q84WU8.p

In [ ]:
!cp /content/embeddings_650M.zip /content/drive/MyDrive/embeddings_650M.zip

In [ ]:
!pip install -q -U bitsandbytes accelerate

In [ ]:
import torch
from transformers import T5EncoderModel, T5Tokenizer, BitsAndBytesConfig
from Bio import SeqIO
from tqdm.auto import tqdm
import os
import shutil
import re
import gc

MODEL_NAME = 'Rostlab/prot_t5_xl_half_uniref50-enc'
SEQ_PATH = 'Train/train_sequences.fasta'
TEST_PATH = 'Test/testsuperset.fasta'
SAVE_DIR = 'embeddings_protT5'

MAX_TOKENS_PER_BATCH = 30000
HARD_LEN_LIMIT = 2048

if os.path.exists(SAVE_DIR): shutil.rmtree(SAVE_DIR)
os.makedirs(SAVE_DIR, exist_ok=True)

device = 'cuda'

print(f"Loading {MODEL_NAME} in 4-bit (NF4)...")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, do_lower_case=False, legacy=False)
model = T5EncoderModel.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)
model.eval()

def get_batches(data, max_tokens):
    batches = []
    current_batch = []

    for item in data:
        current_max_len = max(item['len'], current_batch[0]['len']) if current_batch else item['len']
        current_count = len(current_batch) + 1
        estimated_tokens = current_max_len * current_count

        if estimated_tokens > max_tokens:
            if current_batch:
                batches.append(current_batch)
            current_batch = [item]
        else:
            current_batch.append(item)
    if current_batch:
        batches.append(current_batch)
    return batches

def extract_prott5_ascending(fasta_path, prefix, limit=None):
    print(f"Parsing {prefix}...")
    data = []
    for i, record in enumerate(SeqIO.parse(fasta_path, "fasta")):
        if limit and i >= limit: break

        seq = str(record.seq)
        seq = " ".join(list(re.sub(r"[UZOB]", "X", seq)))

        if len(seq) > HARD_LEN_LIMIT * 2:
             seq = seq[:(HARD_LEN_LIMIT * 2)]

        real_len = len(seq) // 2
        data.append({'id': record.id.split("|")[1] if "|" in record.id else record.id, 'seq': seq, 'len': real_len})

    data.sort(key=lambda x: x['len'], reverse=False)

    batches = get_batches(data, MAX_TOKENS_PER_BATCH)
    print(f"Optimized into {len(batches)} batches (Smallest -> Largest).")

    for batch in tqdm(batches):
        if not batch: continue

        batch_ids = [x['id'] for x in batch]
        batch_seqs = [x['seq'] for x in batch]

        torch.cuda.empty_cache()
        try:
            inputs = tokenizer.batch_encode_plus(batch_seqs, add_special_tokens=True, padding="longest", return_tensors="pt").to(device)
            with torch.inference_mode():
                outputs = model(inputs.input_ids, attention_mask=inputs.attention_mask)
                mask = inputs.attention_mask.unsqueeze(-1)
                embeddings = outputs.last_hidden_state * mask
                embeddings = embeddings.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
                embeddings = embeddings.cpu().half()

            for b in range(len(batch_ids)):
                torch.save(embeddings[b].clone(), os.path.join(SAVE_DIR, f"{prefix}_{batch_ids[b]}.pt"))

        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"OOM on batch. Retrying one-by-one...")
                torch.cuda.empty_cache()
                for item in batch:
                    try:
                        inputs = tokenizer(item['seq'], return_tensors="pt").to(device)
                        with torch.inference_mode():
                            outputs = model(inputs.input_ids, attention_mask=inputs.attention_mask)
                            emb = outputs.last_hidden_state.mean(dim=1).cpu().half()
                            torch.save(emb[0], os.path.join(SAVE_DIR, f"{prefix}_{item['id']}.pt"))
                    except: pass

extract_prott5_ascending(SEQ_PATH, "train", limit=None)
extract_prott5_ascending(TEST_PATH, "test", limit=None)
print("ProtT5 Extraction Complete!")

Loading Rostlab/prot_t5_xl_half_uniref50-enc in 4-bit (NF4)...
Parsing train...
Optimized into 1420 batches (Smallest -> Largest).


  0%|          | 0/1420 [00:00<?, ?it/s]

Parsing test...
Optimized into 3170 batches (Smallest -> Largest).


  0%|          | 0/3170 [00:00<?, ?it/s]

ProtT5 Extraction Complete!


In [ ]:
!zip -r embeddings_protT5.zip /content/embeddings_protT5

Streaming output truncated to the last 5000 lines.
  adding: content/embeddings_protT5/train_Q9JLB5.pt (deflated 29%)
  adding: content/embeddings_protT5/test_Q7Z5L4.pt (deflated 29%)
  adding: content/embeddings_protT5/test_Q58CR3.pt (deflated 29%)
  adding: content/embeddings_protT5/train_Q60775.pt (deflated 29%)
  adding: content/embeddings_protT5/train_Q9XWL9.pt (deflated 29%)
  adding: content/embeddings_protT5/test_P14083.pt (deflated 29%)
  adding: content/embeddings_protT5/test_O70183.pt (deflated 29%)
  adding: content/embeddings_protT5/test_Q8BMC4.pt (deflated 29%)
  adding: content/embeddings_protT5/test_P17481.pt (deflated 29%)
  adding: content/embeddings_protT5/test_B3LNJ6.pt (deflated 29%)
  adding: content/embeddings_protT5/test_Q1AMT3.pt (deflated 29%)
  adding: content/embeddings_protT5/train_O95571.pt (deflated 29%)
  adding: content/embeddings_protT5/test_Q9FYG4.pt (deflated 30%)
  adding: content/embeddings_protT5/train_Q9W0P8.pt (deflated 29%)
  adding: content/em

In [ ]:
!cp /content/embeddings_protT5.zip /content/drive/MyDrive/embeddings_protT5.zip

ESM-2: (Homology)

ProtT5: (Language)

ProstT5/3Di: (Topology)

In [ ]:
import torch
from transformers import T5Tokenizer, T5EncoderModel, BitsAndBytesConfig
from Bio import SeqIO
from tqdm.auto import tqdm
import os
import gc

MODEL_NAME = "Rostlab/ProstT5"
SAVE_DIR = "embeddings_prostT5_3Di"
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 64
CHUNK_SIZE = 50000
MAX_LEN = 1024

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Loading {MODEL_NAME} (Structure) in 4-bit...")

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, do_lower_case=False, legacy=False)
model = T5EncoderModel.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    use_safetensors=True
)
model.eval()
model = torch.compile(model, dynamic=True)

def process_chunk(chunk_data, prefix):
    chunk_data.sort(key=lambda x: len(x['seq']))

    batches = [chunk_data[i:i + BATCH_SIZE] for i in range(0, len(chunk_data), BATCH_SIZE)]

    for batch in tqdm(batches, desc="Encoding Structure", leave=False):
        batch_ids = [x['id'] for x in batch]
        batch_seqs = ["<AA2fold> " + x['seq'] for x in batch]

        try:
            inputs = tokenizer.batch_encode_plus(
                batch_seqs,
                add_special_tokens=True,
                padding="longest",
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt"
            ).to(model.device)

            with torch.inference_mode():
                outputs = model(inputs.input_ids, attention_mask=inputs.attention_mask)

                mask = inputs.attention_mask.unsqueeze(-1)
                embeddings = outputs.last_hidden_state * mask
                embeddings = embeddings.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

                embeddings = embeddings.cpu().half()

            for b, pid in enumerate(batch_ids):
                torch.save(embeddings[b].clone(), os.path.join(SAVE_DIR, f"{prefix}_{pid}.pt"))

        except RuntimeError as e:
            if "out of memory" in str(e):
                print("OOM: Clearing cache.")
                torch.cuda.empty_cache()

def extract_prost_stream(fasta_path, prefix):
    print(f"Streaming {prefix}...")
    chunk_buffer = []

    for record in tqdm(SeqIO.parse(fasta_path, "fasta"), desc="Reading"):
        seq_str = str(record.seq).replace('U','X').replace('Z','X').replace('O','X')
        spaced_seq = " ".join(list(seq_str))
        pid = record.id.split("|")[1] if "|" in record.id else record.id

        chunk_buffer.append({'id': pid, 'seq': spaced_seq})

        if len(chunk_buffer) >= CHUNK_SIZE:
            process_chunk(chunk_buffer, prefix)
            chunk_buffer = []
            gc.collect()

    if chunk_buffer:
        process_chunk(chunk_buffer, prefix)

extract_prost_stream("Train/train_sequences.fasta", "train")
extract_prost_stream("Test/testsuperset.fasta", "test")
print("ProstT5 Extraction Complete.")